# PathoScan — Fine-tuning Gemma 4 E4B on Skin Lesion Data
**Competition:** Gemma 4 Good Hackathon  
**Track:** Health & Sciences + Unsloth Special Prize  
**Model:** google/gemma-4-e4b-it (4B vision-language model)  
**Dataset:** ISIC 2019 Skin Lesion Classification  

This notebook fine-tunes Gemma 4 E4B using Unsloth with LoRA adapters on dermatology image-text pairs.
The resulting model can identify 8 skin lesion types and provide triage recommendations — fully offline.

In [ ]:
# Install dependencies
!pip install unsloth --quiet
!pip install torch torchvision --quiet
!pip install transformers datasets accelerate bitsandbytes --quiet
!pip install pillow pandas scikit-learn tqdm --quiet
print('All dependencies installed.')

In [ ]:
import os
import json
import torch
import pandas as pd
from PIL import Image
from pathlib import Path
from datasets import Dataset
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## Step 1: Load ISIC 2019 Dataset
Download from Kaggle: https://www.kaggle.com/datasets/andrewmvd/isic-2019
Expected structure:
```
/kaggle/input/isic-2019/
  ISIC_2019_Training_Input/   # images
  ISIC_2019_Training_GroundTruth.csv
  ISIC_2019_Training_Metadata.csv
```

In [ ]:
# ISIC 2019 class definitions with clinical descriptions
CLASSES = {
    'MEL':  {
        'name': 'Melanoma',
        'severity': 'HIGH',
        'urgency': 'See a dermatologist within 1 week.',
        'description': 'A malignant melanocytic tumor. Most dangerous form of skin cancer.',
        'action': 'Urgent dermatology referral required. Do not delay.'
    },
    'NV':   {
        'name': 'Melanocytic Nevus (Mole)',
        'severity': 'LOW',
        'urgency': 'Routine annual skin check recommended.',
        'description': 'A benign proliferation of melanocytes (common mole).',
        'action': 'Monitor for changes in size, shape, or color. No immediate action needed.'
    },
    'BCC':  {
        'name': 'Basal Cell Carcinoma',
        'severity': 'MEDIUM',
        'urgency': 'See a doctor within 2–4 weeks.',
        'description': 'The most common type of skin cancer. Rarely metastasizes but can cause local damage.',
        'action': 'Schedule dermatology appointment. Surgical removal is typically curative.'
    },
    'AKIEC':{
        'name': 'Actinic Keratosis / Intraepithelial Carcinoma',
        'severity': 'MEDIUM',
        'urgency': 'See a doctor within 4 weeks.',
        'description': 'A precancerous lesion caused by sun damage. Can progress to squamous cell carcinoma.',
        'action': 'Dermatology consultation for possible cryotherapy or topical treatment.'
    },
    'BKL':  {
        'name': 'Benign Keratosis',
        'severity': 'LOW',
        'urgency': 'Routine check at next doctor visit.',
        'description': 'Includes seborrheic keratosis, solar lentigo, and lichen planus-like keratosis.',
        'action': 'Generally harmless. Removal is cosmetic. Mention to doctor at next visit.'
    },
    'DF':   {
        'name': 'Dermatofibroma',
        'severity': 'LOW',
        'urgency': 'No immediate action needed.',
        'description': 'A benign fibrous skin nodule, commonly found on legs.',
        'action': 'Harmless. No treatment necessary unless symptomatic or cosmetically concerning.'
    },
    'VASC': {
        'name': 'Vascular Lesion',
        'severity': 'LOW',
        'urgency': 'Routine check recommended.',
        'description': 'Includes cherry angiomas, angiokeratomas, and pyogenic granulomas.',
        'action': 'Most are benign. Mention to doctor if bleeding or rapidly growing.'
    },
    'SCC':  {
        'name': 'Squamous Cell Carcinoma',
        'severity': 'HIGH',
        'urgency': 'See a dermatologist within 1–2 weeks.',
        'description': 'A malignant tumor of squamous cells. Can metastasize if untreated.',
        'action': 'Prompt dermatology referral. Biopsy and surgical removal typically required.'
    }
}

print(f'Loaded {len(CLASSES)} skin lesion classes.')

In [ ]:
# Load ground truth CSV
DATA_DIR = Path('/kaggle/input/isic-2019')
IMG_DIR  = DATA_DIR / 'ISIC_2019_Training_Input'

gt_df = pd.read_csv(DATA_DIR / 'ISIC_2019_Training_GroundTruth.csv')

# Convert one-hot to single label
class_cols = list(CLASSES.keys())
gt_df['label'] = gt_df[class_cols].idxmax(axis=1)

print(f'Total samples: {len(gt_df)}')
print('Class distribution:')
print(gt_df['label'].value_counts())

In [ ]:
# Balance dataset: cap each class at 500 samples for training speed
# In a full run you can increase this to 2000+
MAX_PER_CLASS = 500
balanced = gt_df.groupby('label').apply(
    lambda x: x.sample(min(len(x), MAX_PER_CLASS), random_state=42)
).reset_index(drop=True)

# Train/val split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(balanced, test_size=0.1, stratify=balanced['label'], random_state=42)

print(f'Train: {len(train_df)} | Val: {len(val_df)}')

## Step 2: Build Instruction-Tuning Dataset
Each sample = image + instruction prompt + structured JSON response

In [ ]:
SYSTEM_PROMPT = """You are PathoScan, an AI dermatology assistant designed to help 
community health workers in low-resource settings identify and triage skin lesions.

When shown a skin lesion image, analyze it and respond ONLY with a valid JSON object 
in this exact format:
{
  "condition": "<condition name>",
  "condition_code": "<ISIC code>",
  "severity": "LOW|MEDIUM|HIGH",
  "confidence": "<percentage>",
  "description": "<brief clinical description>",
  "urgency": "<timeframe for medical attention>",
  "recommended_action": "<specific next steps>",
  "disclaimer": "This is an AI-assisted screening tool. Always consult a qualified healthcare professional for diagnosis and treatment."
}

Be accurate, concise, and always include the disclaimer."""

USER_PROMPT = "Please analyze this skin lesion image and provide a triage assessment."

def build_sample(row):
    """Build a single instruction-tuning sample from a dataset row."""
    label = row['label']
    info  = CLASSES[label]
    img_path = IMG_DIR / f"{row['image']}.jpg"

    if not img_path.exists():
        return None

    image = Image.open(img_path).convert('RGB').resize((224, 224))

    response = json.dumps({
        'condition':          info['name'],
        'condition_code':     label,
        'severity':           info['severity'],
        'confidence':         '85-92%',  # placeholder; model learns to calibrate
        'description':        info['description'],
        'urgency':            info['urgency'],
        'recommended_action': info['action'],
        'disclaimer':         'This is an AI-assisted screening tool. Always consult a qualified healthcare professional for diagnosis and treatment.'
    }, indent=2)

    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': [
                {'type': 'image', 'image': image},
                {'type': 'text',  'text': USER_PROMPT}
            ]},
            {'role': 'assistant', 'content': response}
        ]
    }

print('Sample builder ready.')

In [ ]:
# Build datasets
from tqdm import tqdm

def build_dataset(df, desc='Building'):
    samples = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        s = build_sample(row)
        if s: samples.append(s)
    return Dataset.from_list(samples)

train_dataset = build_dataset(train_df, 'Building train set')
val_dataset   = build_dataset(val_df,   'Building val set')

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples:   {len(val_dataset)}')

## Step 3: Load Gemma 4 E4B with Unsloth

In [ ]:
MODEL_NAME = 'unsloth/gemma-4-e4b-it-unsloth-bnb-4bit'

model, tokenizer = FastVisionModel.from_pretrained(
    model_name    = MODEL_NAME,
    load_in_4bit  = True,   # 4-bit quantization for memory efficiency
    use_gradient_checkpointing = 'unsloth',
)

print('Model loaded successfully.')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

In [ ]:
# Apply LoRA adapters — fine-tune only a small subset of parameters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r           = 16,    # LoRA rank
    lora_alpha  = 16,
    lora_dropout= 0,
    bias        = 'none',
    random_state= 3407,
    use_rslora  = False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.1f}%)')

## Step 4: Train

In [ ]:
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch = 8
        warmup_steps                = 50,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        eval_strategy               = 'steps',
        eval_steps                  = 100,
        save_strategy               = 'steps',
        save_steps                  = 100,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 3407,
        output_dir                  = '/kaggle/working/pathoscan-checkpoints',
        report_to                   = 'none',
        remove_unused_columns       = False,
        dataset_text_field          = '',
        dataset_kwargs              = {'skip_prepare_dataset': True},
        max_seq_length              = 2048,
    ),
)

print('Trainer configured. Starting training...')
trainer_stats = trainer.train()
print(f'Training complete! Runtime: {trainer_stats.metrics["train_runtime"]:.0f}s')

## Step 5: Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained('/kaggle/working/pathoscan-lora')
tokenizer.save_pretrained('/kaggle/working/pathoscan-lora')
print('LoRA adapters saved.')

# Save merged 16-bit model for Ollama
model.save_pretrained_merged(
    '/kaggle/working/pathoscan-merged',
    tokenizer,
    save_method='merged_16bit'
)
print('Merged 16-bit model saved.')

# Save GGUF 4-bit quantized for Ollama local deployment
model.save_pretrained_gguf(
    '/kaggle/working/pathoscan-gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
print('GGUF (Q4_K_M) model saved — ready for Ollama!')

In [ ]:
# Quick inference test
FastVisionModel.for_inference(model)

test_row = val_df.sample(1).iloc[0]
test_img = Image.open(IMG_DIR / f"{test_row['image']}.jpg").convert('RGB')

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user',   'content': [
        {'type': 'image', 'image': test_img},
        {'type': 'text',  'text': USER_PROMPT}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(test_img, input_text, return_tensors='pt').to('cuda')

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=512, temperature=0.1)

response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print(f'\nGround truth: {test_row["label"]} ({CLASSES[test_row["label"]]["name"]})')
print(f'\nModel response:\n{response}')

# Validate JSON
try:
    parsed = json.loads(response)
    print('\nJSON is valid!')
    print(f'Predicted: {parsed["condition"]} | Severity: {parsed["severity"]}')
except:
    print('Warning: response is not valid JSON. May need more training.')

## Step 6: Push to HuggingFace Hub (optional but recommended)
This makes your model publicly accessible for the competition submission.

In [ ]:
# Set your HuggingFace token first:
# import os; os.environ['HF_TOKEN'] = 'your_token_here'

HF_USERNAME = 'YOUR_HF_USERNAME'  # replace with your HuggingFace username

model.push_to_hub_gguf(
    f'{HF_USERNAME}/pathoscan-gemma4-e4b',
    tokenizer,
    quantization_method = ['q4_k_m', 'q8_0'],
    token = os.environ.get('HF_TOKEN')
)
print(f'Model pushed to: https://huggingface.co/{HF_USERNAME}/pathoscan-gemma4-e4b')